<a href="https://colab.research.google.com/github/Rxv3640/AI-Product-Review-Analyzer/blob/main/analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Testing pretrained sentiment model using pipeline() API**

In [16]:
from transformers import pipeline
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "fancyzhx/amazon_polarity",
    split = {
        "train": "train[:5]",
        "test": "test[:5]"
    }

)

test_reviews = dataset["train"]["content"]

pipeline_model = pipeline("sentiment-analysis")

for review in test_reviews:
  print(pipeline_model(review))


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9991727471351624}]
[{'label': 'POSITIVE', 'score': 0.9938104748725891}]
[{'label': 'POSITIVE', 'score': 0.9997978806495667}]
[{'label': 'POSITIVE', 'score': 0.9998179078102112}]
[{'label': 'POSITIVE', 'score': 0.9997859597206116}]


**Fine-Tuning**

In [14]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer
from transformers import DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

training_args = TrainingArguments("test-trainer")

def tokenize_function(dataset):
  return tokenizer(dataset["content"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator = DataCollatorWithPadding(tokenizer),
    processing_class=tokenizer,
)

trainer.train()







Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=0.4965078830718994, metrics={'train_runtime': 31.3992, 'train_samples_per_second': 0.478, 'train_steps_per_second': 0.096, 'total_flos': 745129117440.0, 'train_loss': 0.4965078830718994, 'epoch': 3.0})

**Evaluation**

In [20]:
predictions = trainer.predict(tokenized_dataset["test"])

import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

import evaluate

metric = evaluate.load("glue", "mrpc")

print(metric.compute(predictions=preds, references=predictions.label_ids))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'accuracy': 0.8, 'f1': 0.8888888888888888}
